## Notebook42

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
! pip install umap-learn --quiet
! python -m spacy download en_core_web_sm

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c
import numpy as np

import funs
from funs import DSText, umap_text

import spacy

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
docs = (
    pl.read_parquet(ub + "data/agnews_pca.parquet")
    .filter(c.index == "test")
    .select(c.id, c.label, c.text)
    .rename({"id": "doc_id"})
    .sort(c.doc_id)
    .group_by(c.label, maintain_order=True)
    .head(50)
)

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
anno = DSText.process(docs.select(c.doc_id, c.text), nlp)

### Questions

`docs` and `anno` hold the same 200 AG News articles as the last notebook, 50 each
labeled `"Business"`, `"Sci/Tech"`, `"Sports"`, and `"World"`.

Unless a question says otherwise, just print the result of each block; do not save
it to a variable.

1. Compute the TF-IDF scores with
`DSText.compute_tfidf(anno, min_df=0.01, max_df=0.5)` and print the result.

In [ ]:
DSText.compute_tfidf(anno, min_df=0.01, max_df=0.5)

2. Take the code from the previous question and keep only the rows where `lemma` is
`"olympic"`. Sort from the largest to the smallest `tfidf`.

In [ ]:
(
    DSText.compute_tfidf(anno, min_df=0.01, max_df=0.5)
    .filter(c.lemma == "olympic")
    .sort(c.tfidf, descending=True)
)

Only 11 of the 200 articles use the word "olympic". Most articles have no row
here at all, because the TF-IDF table only stores the scores that are not zero.

3. Change the code from the previous question to keep the lemmas `"olympic"` and
`"profit"`. Then, as in the chapter, pivot the result so that each lemma is a
column, using `.pivot(on="lemma", index="doc_id", values="tfidf")`, and fill the
missing values with zero using `.fill_null(0)`.

In [ ]:
(
    DSText.compute_tfidf(anno, min_df=0.01, max_df=0.5)
    .filter(c.lemma.is_in(["olympic", "profit"]))
    .pivot(on="lemma", index="doc_id", values="tfidf")
    .fill_null(0)
)

4. Take the code from the previous question and draw a scatter plot with
`geom_point`, with `olympic` on the x-axis and `profit` on the y-axis.

In [ ]:
(
    DSText.compute_tfidf(anno, min_df=0.01, max_df=0.5)
    .filter(c.lemma.is_in(["olympic", "profit"]))
    .pivot(on="lemma", index="doc_id", values="tfidf")
    .fill_null(0)
    .ggplot(aes(c.olympic, c.profit))
    .geom_point()
)

No article uses both words. The points lie along the two axes, and the other 178
articles, which use neither word, are not in the table at all. `olympic` and
`profit` come from unrelated topics, so no article has a reason to use both.

5. Build a 2D UMAP projection of the articles, as in the chapter. Pipe `anno` into
`umap_text` with `doc_id=c.doc_id`, `term_id=c.lemma`, `n_components=2`,
`min_df=0.01`, `max_df=0.5`, and `random_state=42`, and then call
`.predict(full=True)`.

In [ ]:
(
    anno
    .pipe(
        umap_text,
        doc_id=c.doc_id,
        term_id=c.lemma,
        n_components=2,
        min_df=0.01,
        max_df=0.5,
        random_state=42
    )
    .predict(full=True)
)

6. Take the code from the previous question, join it to
`docs.select(c.doc_id, c.label)`, and draw a scatter plot with `dr0` on the x-axis
and `dr1` on the y-axis, colored by `label`.

In [ ]:
(
    anno
    .pipe(
        umap_text,
        doc_id=c.doc_id,
        term_id=c.lemma,
        n_components=2,
        min_df=0.01,
        max_df=0.5,
        random_state=42
    )
    .predict(full=True)
    .join(docs.select(c.doc_id, c.label), on=c.doc_id)
    .ggplot(aes(c.dr0, c.dr1, color=c.label))
    .geom_point()
)

The labels form loose groups, such as the Sports articles near the top, but the
four topics overlap a lot. The chapter's author pages separated much more cleanly.

7. The block below is given. It computes the angle distance between every pair of
articles with `DSText.compute_distances()` and then, as in the chapter, stacks the
table with a copy of itself with the two ID columns swapped. The result is saved as
`text_dist_both`. Run it and look at the table.

In [ ]:
text_dist = DSText.compute_distances(anno, min_df=0.01, max_df=0.5)

text_dist_both = pl.concat([
    text_dist,
    text_dist.select(doc_id=c.doc_id2, doc_id2=c.doc_id, distance=c.distance)
])
text_dist_both

8. The article `doc120027` is about the swimmer Michael Phelps winning a gold medal.
Filter `text_dist_both` to this article, sort by `distance`, and keep the five
closest matches.

In [ ]:
(
    text_dist_both
    .filter(c.doc_id == "doc120027")
    .sort(c.distance)
    .head(5)
)

The closest match, `doc120159`, is another article about Phelps at the same
Olympics. The other four are further away and are about other Olympic events.

9. Find the nearest neighbor of every article, as in the chapter: sort by `doc_id`
and `distance`, and keep the first row of each `doc_id` with
`.group_by(c.doc_id, maintain_order=True).head(1)`.

In [ ]:
(
    text_dist_both
    .sort(c.doc_id, c.distance)
    .group_by(c.doc_id, maintain_order=True)
    .head(1)
)

10. Sort `text_dist_both` from the smallest to the largest `distance` and keep the
first four rows.

In [ ]:
(
    text_dist_both
    .sort(c.distance)
    .head(4)
)

Each pair appears twice, once in each direction. The closest pair, `doc120195`
and `doc120199`, has a distance of 0.37, far below the next pair at 0.77.

11. Filter `docs` to the two articles in the closest pair, `"doc120195"` and
`"doc120199"`. Select the `text` column and print the full text with
`.to_series().to_list()`.

In [ ]:
(
    docs
    .filter(c.doc_id.is_in(["doc120195", "doc120199"]))
    .select(c.text)
    .to_series()
    .to_list()
)

The two articles are versions of the same Reuters story about Medtronic's
earnings. Apart from the headline, they differ by a single word ("amid" and "on").